# 03 — Part E: Development of Three Base Models

**6001CEM Machine Learning Group Project — Diabetes 130-US Hospitals Readmission Prediction**

**Inputs (produced in `02_Data_Preprocessing.ipynb`, see Project Handover D012–D020):**
- `data/processed/train_clean_selected.csv` (81,526 rows × 46 cols incl. target)
- `data/processed/test_clean_selected.csv` (20,237 rows × 46 cols incl. target)
- Target column: `readmitted_30` (1 = readmitted within 30 days, 0 = not)

**Known constraint carried over from Part D (D018):** class-imbalance handling
(none / class-weighting / SMOTE) must be compared **experimentally**, using an
`imblearn.pipeline.Pipeline` so resampling is fitted only within CV training folds —
never applied to the saved CSVs directly.

**Documented limitation (read before running):** `patient_nbr` was dropped from the
feature-selected export (D006/D019), so cross-validation *within* the training set is
a plain `StratifiedKFold`, not grouped by patient. This does **not** violate the
train/test leakage guarantee from D016 (zero patient overlap between train and test is
already verified) — but it means a given patient's multiple encounters could land in
different CV folds, which can make CV scores marginally optimistic. This is disclosed
here rather than hidden, and should be stated explicitly in the Part E/F report text as
an acknowledged limitation, not silently assumed away.

**Structure of this notebook (Observation → Evidence → Interpretation → Decision):**
1. Load data, sanity checks
2. Baseline reference point (majority-class dummy classifier)
3. Model candidate justification (why these three, not just because they're conventional)
4. Imbalance-strategy comparison per model (none vs class_weight vs SMOTE) via CV
5. Fit each model's chosen configuration, evaluate on held-out test set
6. Consolidated comparison table + discussion (feeds Part F and Part H)


In [1]:
pip install imblearn xgboost --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score,
    confusion_matrix, classification_report, make_scorer
)

# imblearn's Pipeline (not sklearn's) is required so that SMOTE is fitted
# only within each CV training fold, never on the validation fold or test set.
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_XGB = False
    print("xgboost not installed — falling back to sklearn GradientBoostingClassifier. "
          "Install with: pip install xgboost --break-system-packages")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data/processed")
TARGET = "readmitted_30"


## 1. Load Data and Sanity Checks

Reproducing the exact class-balance figures from D016 confirms we are working from the
correct, already-cleaned files and haven't accidentally introduced a preprocessing
regression before modelling begins.


In [3]:
train_df = pd.read_csv("../data/processed/train_clean_selected.csv")
test_df = pd.read_csv("../data/processed/test_clean_selected.csv")

print(f"Train shape: {train_df.shape}   (expected 81,526 x 46)")
print(f"Test shape:  {test_df.shape}   (expected 20,237 x 46)")

assert TARGET in train_df.columns and TARGET in test_df.columns, "Target column missing"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

train_balance = y_train.value_counts(normalize=True).round(4) * 100
test_balance = y_test.value_counts(normalize=True).round(4) * 100
print(f"\nTrain class balance (%):\n{train_balance}")
print(f"\nTest class balance (%):\n{test_balance}")
print("\nExpected (from D016): Train 88.76% / 11.24%,  Test 89.18% / 10.82%")

assert X_train.shape[1] == 45, f"Expected 45 features, got {X_train.shape[1]}"


Train shape: (81526, 46)   (expected 81,526 x 46)
Test shape:  (20237, 46)   (expected 20,237 x 46)

Train class balance (%):
readmitted_30
0    88.76
1    11.24
Name: proportion, dtype: float64

Test class balance (%):
readmitted_30
0    89.18
1    10.82
Name: proportion, dtype: float64

Expected (from D016): Train 88.76% / 11.24%,  Test 89.18% / 10.82%


## 2. Reference Baseline

Before evaluating real models, establish what a trivial/naive classifier achieves.
This is the number every candidate model must clearly beat to be worth reporting
(D009: a majority-class classifier reaches ~88.8% *accuracy* purely by predicting
"not readmitted" every time — which is exactly why accuracy alone is inappropriate here,
and why PR-AUC / recall on the minority class are treated as the primary metrics
throughout this notebook).


In [4]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_proba = dummy.predict_proba(X_test)[:, 1]

print("Majority-class dummy baseline (test set):")
print(f"  Accuracy : {accuracy_score(y_test, dummy_pred):.4f}")
print(f"  Recall   : {recall_score(y_test, dummy_pred, zero_division=0):.4f}  <- will always be 0")
print(f"  F1       : {f1_score(y_test, dummy_pred, zero_division=0):.4f}")
print(f"  PR-AUC   : {average_precision_score(y_test, dummy_proba):.4f}  (~= positive class prevalence)")
print(f"  ROC-AUC  : {roc_auc_score(y_test, dummy_proba):.4f}  <- uninformative for a constant predictor")


Majority-class dummy baseline (test set):
  Accuracy : 0.8918
  Recall   : 0.0000  <- will always be 0
  F1       : 0.0000
  PR-AUC   : 0.1082  (~= positive class prevalence)
  ROC-AUC  : 0.5000  <- uninformative for a constant predictor


## 3. Candidate Model Selection and Justification

Per the assignment brief and Responsibility.txt, exactly **three** classification models
are required (Person A: Models 1–2, Person B: Model 3). The candidates below are chosen
against explicit criteria rather than by default popularity, as the handover doc requires.

| Criterion | Logistic Regression | Random Forest | Gradient Boosting (XGBoost) |
|---|---|---|---|
| Algorithm family | Linear, parametric | Bagging ensemble of trees | Boosting ensemble of trees |
| Key assumption | Linear decision boundary in log-odds space; feature independence not required but multicollinearity affects coefficient stability | No distributional assumptions; handles non-linear interactions | No distributional assumptions; sequential error-correction |
| Interpretability | High — coefficients map directly to named one-hot/ordinal features (D020's stated reason for rejecting PCA) | Medium — feature importances, but less directly attributable than coefficients | Medium — feature importances / SHAP possible, but less transparent than logistic regression |
| Handles non-linear interactions | No (unless engineered) | Yes, natively | Yes, natively, typically stronger than RF on tabular data |
| Computational cost | Low | Medium | Medium–High (mitigated by `n_jobs`, early stopping) |
| Native class-imbalance handling | `class_weight='balanced'` | `class_weight='balanced'` | `scale_pos_weight` |
| Suitability for healthcare decision support | Strong — clinicians/administrators can audit *why* a case was flagged | Moderate — robust, but less directly explainable | Moderate — typically best raw discriminative performance, but least transparent of the three |

**Rationale for this exact set of three:**

1. **Logistic Regression** — serves as an interpretable, computationally cheap linear
   benchmark appropriate for a healthcare deployment context (Part H explicitly requires
   discussion of interpretability and deployment suitability). It also directly reuses the
   interpretability argument already established in D020 (PCA rejected specifically to
   preserve named, attributable features).
2. **Random Forest** — a non-linear ensemble baseline capable of capturing feature
   interactions (e.g. `diag_1_group` × `age`) that logistic regression cannot, while
   remaining relatively robust to hyperparameter misspecification and to the mixed
   one-hot/ordinal/numeric feature space produced in D017.
3. **Gradient Boosting (XGBoost)** — typically the strongest tabular-data performer among
   the three, included specifically to test whether the extra model complexity and
   sequential error-correction meaningfully outperforms Random Forest on this dataset, or
   whether the added complexity is not justified by the evidence — this experimental
   comparison is exactly what determines the Part G ensemble selection (best two of three)
   and cannot be assumed in advance.

**Alternative considered and rejected:** Support Vector Machines were considered but
rejected on computational-cost grounds — SVMs scale poorly (roughly quadratic-to-cubic)
with ~81,500 training rows and would require kernel-parameter search on top of the
class-imbalance comparison already required by D018, which was judged not to be a good
use of the assignment's scope relative to its expected benefit over the three selected
models. This trade-off should be stated explicitly in the report.


## 4. Class-Imbalance Strategy Comparison (per model, via CV)

For each of the three models, three imbalance strategies are compared:

- **`none`** — model trained as-is (imbalance ignored)
- **`class_weight`** — model's native `class_weight='balanced'` / `scale_pos_weight`
- **`smote`** — `SMOTE` oversampling of the minority class, wrapped inside an
  `imblearn.pipeline.Pipeline` so it is fit **only** on each CV training fold

Primary metric: **PR-AUC (Average Precision)**, because with an 11.16% positive class,
ROC-AUC can look deceptively strong while precision on the minority class remains poor
(D009). Recall and F1 are reported alongside for the healthcare-context discussion in
Part H (false negatives — missed at-risk patients — vs false positives).

**5-fold `StratifiedKFold`** is used (not grouped by patient — see the limitation noted
in Section 1 above).


In [6]:
from pathlib import Path
Path("results").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
}

def build_model(name, strategy):
    """Return an unfitted classifier configured for the given imbalance strategy.
    'smote' strategy returns None here — SMOTE is added at the pipeline level instead."""
    if name == "logreg":
        cw = "balanced" if strategy == "class_weight" else None
        return LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight=cw)
    if name == "rf":
        cw = "balanced" if strategy == "class_weight" else None
        return RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight=cw
        )
    if name == "xgb":
        if HAS_XGB:
            spw = None
            if strategy == "class_weight":
                spw = (y_train == 0).sum() / (y_train == 1).sum()
            return XGBClassifier(
                n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1,
                eval_metric="logloss", scale_pos_weight=spw,
            )
        else:
            return GradientBoostingClassifier(random_state=RANDOM_STATE)
    raise ValueError(name)

def build_pipeline(name, strategy):
    est = build_model(name, strategy)
    if strategy == "smote":
        return ImbPipeline([
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("clf", est),
        ])
    return ImbPipeline([("clf", est)])

model_names = ["logreg", "rf", "xgb"]
strategies = ["none", "class_weight", "smote"]

results = []
for m in model_names:
    for s in strategies:
        pipe = build_pipeline(m, s)
        cv_res = cross_validate(
            pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, error_score="raise"
        )
        row = {"model": m, "strategy": s}
        for metric in scoring:
            row[f"cv_{metric}_mean"] = cv_res[f"test_{metric}"].mean()
            row[f"cv_{metric}_std"] = cv_res[f"test_{metric}"].std()
        results.append(row)
        print(f"{m:8s} | {s:12s} | PR-AUC {row['cv_pr_auc_mean']:.4f} "
              f"| ROC-AUC {row['cv_roc_auc_mean']:.4f} | Recall {row['cv_recall_mean']:.4f} "
              f"| F1 {row['cv_f1_mean']:.4f}")

cv_results_df = pd.DataFrame(results)
cv_results_df.to_csv("results/cv_imbalance_comparison.csv", index=False)
cv_results_df


logreg   | none         | PR-AUC 0.1918 | ROC-AUC 0.6251 | Recall 0.0156 | F1 0.0302
logreg   | class_weight | PR-AUC 0.1918 | ROC-AUC 0.6268 | Recall 0.4581 | F1 0.2488
logreg   | smote        | PR-AUC 0.1881 | ROC-AUC 0.6167 | Recall 0.5209 | F1 0.2426
rf       | none         | PR-AUC 0.1440 | ROC-AUC 0.5723 | Recall 0.0403 | F1 0.0667
rf       | class_weight | PR-AUC 0.1271 | ROC-AUC 0.5429 | Recall 0.1544 | F1 0.1395
rf       | smote        | PR-AUC 0.1343 | ROC-AUC 0.5587 | Recall 0.1209 | F1 0.1311
xgb      | none         | PR-AUC 0.1664 | ROC-AUC 0.5959 | Recall 0.0239 | F1 0.0445
xgb      | class_weight | PR-AUC 0.1588 | ROC-AUC 0.5769 | Recall 0.4221 | F1 0.2200
xgb      | smote        | PR-AUC 0.1428 | ROC-AUC 0.5628 | Recall 0.0746 | F1 0.1029


,model,strategy,cv_pr_auc_mean,cv_pr_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_recall_mean,cv_recall_std,cv_precision_mean,cv_precision_std
0,logreg,none,0.191832,0.008757,0.625128,0.005056,0.030178,0.007038,0.015600,0.003693,0.470000,0.054910
1,logreg,class_weight,0.191834,0.009037,0.626795,0.005756,0.248751,0.005575,0.458058,0.009957,0.170738,0.003941
2,logreg,smote,0.188088,0.008712,0.616674,0.005155,0.242615,0.004246,0.520893,0.011185,0.158142,0.002753
3,rf,none,0.144007,0.003235,0.572271,0.004374,0.066728,0.004547,0.040253,0.002631,0.195159,0.016938
4,rf,class_weight,0.127063,0.002590,0.542919,0.004877,0.139488,0.007025,0.154357,0.009381,0.127291,0.005760
5,rf,smote,0.134258,0.002287,0.558734,0.003221,0.131087,0.009681,0.120867,0.011645,0.143453,0.007859
6,xgb,none,0.166420,0.003270,0.595866,0.003119,0.044542,0.001090,0.023890,0.000631,0.331205,0.026527
7,xgb,class_weight,0.158803,0.002456,0.576921,0.006978,0.220017,0.001950,0.422056,0.007693,0.148813,0.001754
8,xgb,smote,0.142816,0.001343,0.562762,0.004298,0.102888,0.002954,0.074614,0.004499,0.167310,0.010827


### Interpreting the CV comparison

**Fill this in after running the cell above with the actual numbers** — per the project's
experimental-results policy (handover Section 23), do not write conclusions here until the
CV has actually been executed. Suggested structure per model:

> *[Model] achieved its highest CV PR-AUC of [X] under the [strategy] configuration,
> compared to [Y] (none) and [Z] (the alternative). This is consistent with / contrary to
> the expectation that [reasoning], because [evidence from the numbers]. The trade-off
> observed was [e.g. recall improved but precision fell by...]. Given the healthcare
> context — where a missed at-risk patient (false negative) is arguably more costly than
> an unnecessary follow-up call (false positive) — [chosen strategy] is selected for
* [Model] going forward, because [evidence-based reason].*

Record the **selected strategy per model** in the dictionary below once the CV results are in.


In [10]:
# --- FILL IN after inspecting cv_results_df above ---
selected_strategy = {
    "logreg": "class_weight", 
    "rf": "class_weight",
    "xgb": "class_weight",
}
assert all(v is not None for v in selected_strategy.values()), (
    "Set the selected imbalance strategy for each model based on the CV results "
    "before proceeding — do not hard-code a guess."
)


## 5. Fit Final Baseline Configuration per Model, Evaluate on Held-Out Test Set

Each model is now fit **once** on the full training set using its selected imbalance
strategy, and evaluated on the untouched test set (D024: the test set is touched only
here, for the first time, preserving the leakage policy).

These are **baseline** (non-tuned) results — hyperparameter optimisation is Part F, not
this notebook. Recording baseline numbers here is required so that Part F can honestly
report "performance before vs after optimisation" per the assignment brief.


In [12]:
def evaluate_on_test(name, pipe, threshold=0.5):
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
    }
    cm = confusion_matrix(y_test, pred)
    print(f"=== {name} ===")
    metrics_for_print = {k: v for k, v in metrics.items() if k != "model"}
    print(pd.Series(metrics_for_print).round(4))
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    print(classification_report(y_test, pred, zero_division=0))
    print()
    return pipe, metrics, cm

fitted_models = {}
baseline_metrics = []

for m in model_names:
    pipe = build_pipeline(m, selected_strategy[m])
    fitted_pipe, metrics, cm = evaluate_on_test(m, pipe)
    fitted_models[m] = fitted_pipe
    metrics["imbalance_strategy"] = selected_strategy[m]
    baseline_metrics.append(metrics)

baseline_df = pd.DataFrame(baseline_metrics)
baseline_df.to_csv("results/baseline_model_comparison.csv", index=False)
baseline_df


=== logreg ===
accuracy     0.6941
precision    0.1639
recall       0.4452
f1           0.2396
roc_auc      0.6235
pr_auc       0.1911
dtype: float64
Confusion matrix [[TN, FP], [FN, TP]]:
[[13072  4975]
 [ 1215   975]]
              precision    recall  f1-score   support

           0       0.91      0.72      0.81     18047
           1       0.16      0.45      0.24      2190

    accuracy                           0.69     20237
   macro avg       0.54      0.58      0.52     20237
weighted avg       0.83      0.69      0.75     20237


=== rf ===
accuracy     0.7830
precision    0.1198
recall       0.1584
f1           0.1365
roc_auc      0.5356
pr_auc       0.1205
dtype: float64
Confusion matrix [[TN, FP], [FN, TP]]:
[[15498  2549]
 [ 1843   347]]
              precision    recall  f1-score   support

           0       0.89      0.86      0.88     18047
           1       0.12      0.16      0.14      2190

    accuracy                           0.78     20237
   macro avg      

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,imbalance_strategy
0,logreg,0.694125,0.163866,0.445205,0.239558,0.623485,0.191133,class_weight
1,rf,0.782972,0.119820,0.158447,0.136453,0.535644,0.120487,class_weight
2,xgb,0.671443,0.147733,0.426941,0.219509,0.587294,0.166617,class_weight


## 6. Feature Importance / Coefficients (supports Part H interpretability discussion)

Reusing the D020 argument: because PCA was rejected in favour of retaining named,
one-hot/ordinal features, every model here can attribute a prediction to a specific
clinical factor. This section extracts that attribution for each model as evidence for
Part H.


In [13]:
feature_names = X_train.columns.tolist()

# Logistic Regression coefficients
logreg_clf = fitted_models["logreg"].named_steps["clf"]
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": logreg_clf.coef_[0],
}).sort_values("coefficient", key=abs, ascending=False)
print("Top 10 Logistic Regression coefficients by magnitude:")
print(coef_df.head(10).to_string(index=False))

# Random Forest importances
rf_clf = fitted_models["rf"].named_steps["clf"]
rf_imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_clf.feature_importances_,
}).sort_values("importance", ascending=False)
print("\nTop 10 Random Forest feature importances:")
print(rf_imp_df.head(10).to_string(index=False))

# XGBoost / GBM importances
xgb_clf = fitted_models["xgb"].named_steps["clf"]
xgb_imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_clf.feature_importances_,
}).sort_values("importance", ascending=False)
print("\nTop 10 Gradient Boosting feature importances:")
print(xgb_imp_df.head(10).to_string(index=False))


Top 10 Logistic Regression coefficients by magnitude:
                                          feature  coefficient
                            num__number_inpatient     0.387887
                    nom__diag_1_group_Respiratory    -0.303698
               nom__medical_specialty_Orthopedics     0.273405
                       nom__admission_source_id_6    -0.261800
                         nom__admission_type_id_6     0.260263
                nom__medical_specialty_Cardiology    -0.255988
nom__medical_specialty_Orthopedics-Reconstructive    -0.251543
                       nom__admission_source_id_2    -0.249329
                         nom__admission_type_id_2     0.236891
                       nom__admission_source_id_4    -0.229086

Top 10 Random Forest feature importances:
                                feature  importance
                               ord__age    0.210505
                  num__number_inpatient    0.131679
                 num__number_outpatient    0.073543
  

## 7. Save Artifacts for Part F (Hyperparameter Optimisation) and Part G (Ensemble)

Persisting the fitted baseline pipelines and metrics so Part F can load them directly
rather than re-running Part E, and so Part F's "before optimisation" numbers are
guaranteed to match what is actually reported here (avoiding the exact pitfall the
handover doc warns against in Section 23 — never write results before they exist).


In [14]:
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

for name, pipe in fitted_models.items():
    joblib.dump(pipe, f"models/baseline_{name}.joblib")

print("Saved baseline pipelines to models/baseline_{logreg,rf,xgb}.joblib")
print("Saved results/cv_imbalance_comparison.csv and results/baseline_model_comparison.csv")
print()
print("Baseline summary (test set):")
print(baseline_df[["model", "imbalance_strategy", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]])


Saved baseline pipelines to models/baseline_{logreg,rf,xgb}.joblib
Saved results/cv_imbalance_comparison.csv and results/baseline_model_comparison.csv

Baseline summary (test set):
    model imbalance_strategy  accuracy  precision    recall        f1  \
0  logreg       class_weight  0.694125   0.163866  0.445205  0.239558   
1      rf       class_weight  0.782972   0.119820  0.158447  0.136453   
2     xgb       class_weight  0.671443   0.147733  0.426941  0.219509   

    roc_auc    pr_auc  
0  0.623485  0.191133  
1  0.535644  0.120487  
2  0.587294  0.166617  


## 8. Status and Next Steps

**What this notebook has done:**
- Loaded and sanity-checked the Part D outputs
- Established a naive-baseline reference point (D009's accuracy trap made explicit)
- Justified the three model choices against explicit criteria (not just convention)
- Compared three imbalance-handling strategies per model via CV, inside a leakage-safe `imblearn.pipeline.Pipeline` (D018)
- Fitted each model's selected configuration once, evaluated once on the untouched test set (D024)
- Extracted feature importances/coefficients for the Part H interpretability discussion (reusing D020's reasoning)
- Saved fitted pipelines for Part F to load directly

**What remains (not in scope for this notebook):**
- Part F: hyperparameter search (Grid/Random Search + CV) on top of the imbalance strategy already selected here
- Part G: select best two of these three models for the ensemble, using the *tuned* Part F results, not these baseline numbers
- Part H: full comparative evaluation (baseline vs tuned vs ensemble)
- Confirm D014 (`diag_1_group`) with the full team before this feature is presented as agreed methodology at viva

**Reminder before writing this up in the report:** do not report the numbers in
`baseline_model_comparison.csv` as final results — these are Part E baselines only, and
the report should distinguish clearly between "baseline" and "optimised" (Part F) performance.
